In [ ]:
#STEP 1

from __future__ import annotations

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import os
import re
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("OpenPyXL:", openpyxl.__version__)

In [ ]:
# STEP 2--------------------------- EDITABLE PATH SETTINGS ---------------------------
RESEARCH_ROOT = Path(
    os.environ.get("RESEARCH_ANALYSIS_ROOT", r"Z:\Research-analysis")
)
NOTEBOOK_NAME = "SNES-ECNES-data"
NOTEBOOK_DIR = RESEARCH_ROOT / "notebook"
NOTES_DIR = RESEARCH_ROOT / "0-Notes"
RAW_TRANSFER_ZIP = Path(
    os.environ.get(
        "LUTO_TRANSFER_ZIP",
        str(
            RESEARCH_ROOT
            / "data"
            / "raw"
            / "LUTO2-Scenario-Run-2"
            / "transfer_3573951_files_89a7061e.zip"
        ),
    )
)
FILE_INVENTORY_PATH = Path(
    os.environ.get(
        "LUTO_FILE_INVENTORY",
        str(NOTES_DIR / "Combined_file_inventory.xlsx"),
    )
)

# ------------------------- EDITABLE COVERAGE SETTINGS -------------------------
RUN_IDS = list(range(1, 17))
RUN_CODES = [f"Run_G{run_id:04d}" for run_id in RUN_IDS]
START_YEAR = 2025
END_YEAR = 2050
EXPECTED_YEARS = list(range(START_YEAR, END_YEAR + 1, 5))
YEAR_SHEET_NAMES = [str(year) for year in EXPECTED_YEARS]
CSV_CHUNK_ROWS = 100_000

# Excel allows 1,048,576 total rows; one row is reserved for the header.
MAX_DATA_ROWS_PER_SHEET = 1_048_575
MAX_EXCEL_COLUMNS = 16_384

# ------------------------------ OUTPUT PATHS ----------------------------------
OUTPUT_DIR = RESEARCH_ROOT / "data" / "processed" / NOTEBOOK_NAME
TEMP_DIR = OUTPUT_DIR / "temp"
RUN_WORKBOOK_PATHS = {
    run_code: OUTPUT_DIR / f"SNES-ECNES-{run_code}.xlsx"
    for run_code in RUN_CODES
}
INVENTORY_SELECTION_CSV = OUTPUT_DIR / f"{NOTEBOOK_NAME}_inventory_selection.csv"
SOURCE_AUDIT_CSV = OUTPUT_DIR / f"{NOTEBOOK_NAME}_source_audit.csv"
SHEET_AUDIT_CSV = OUTPUT_DIR / f"{NOTEBOOK_NAME}_sheet_audit.csv"
VALIDATION_CSV = OUTPUT_DIR / f"{NOTEBOOK_NAME}_validation.csv"

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("Runs:", RUN_CODES[0], "to", RUN_CODES[-1])
print("Year sheets in every workbook:", YEAR_SHEET_NAMES)
print("Output folder:", OUTPUT_DIR)
print("Example workbook:", RUN_WORKBOOK_PATHS[RUN_CODES[0]])
print("Temporary folder:", TEMP_DIR)

In [ ]:
# STEP 3

FILE_FAMILIES = {
    "GBF2_priority_scores": re.compile(
        r"^biodiversity_GBF2_priority_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "GBF4_ECNES_scores": re.compile(
        r"^biodiversity_GBF4_ECNES_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "GBF4_ECNES_sum_scores": re.compile(
        r"^biodiversity_GBF4_ECNES_sum_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "GBF4_SNES_scores": re.compile(
        r"^biodiversity_GBF4_SNES_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "GBF4_SNES_sum_scores": re.compile(
        r"^biodiversity_GBF4_SNES_sum_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "overall_priority_scores": re.compile(
        r"^biodiversity_overall_priority_scores_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
    "overall_priority_scores_all": re.compile(
        r"^biodiversity_overall_priority_scores_all_(\d{4})\.csv$",
        flags=re.IGNORECASE,
    ),
}

PROVENANCE_COLUMNS = [
    "Run_ID",
    "Run_Name",
    "Dataset_Family",
    "File_Year",
    "Source_File",
    "Source_Member",
    "Source_Row",
]

EXPECTED_FILE_KEYS = {
    (family, year)
    for family in FILE_FAMILIES
    for year in EXPECTED_YEARS
}
EXPECTED_FILES_PER_RUN = len(EXPECTED_FILE_KEYS)

print("File families:", list(FILE_FAMILIES))
print("Expected files per run:", EXPECTED_FILES_PER_RUN)


In [ ]:
# STEP 4

def normalise_name(value):
    # Convert labels into a compact key for case-insensitive comparisons.
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().casefold())


def normalise_zip_path(value):
    # ZIP paths always use forward slashes, including on Windows.
    return str(PurePosixPath(str(value).replace("\\", "/"))).casefold()


def resolve_zip_member(zip_file, requested):
    # Match a complete member first, then a unique suffix/member filename.
    requested_key = normalise_zip_path(requested)
    member_map = {
        normalise_zip_path(name): name
        for name in zip_file.namelist()
        if not name.endswith("/")
    }
    if requested_key in member_map:
        return member_map[requested_key]
    suffix_matches = [
        original
        for key, original in member_map.items()
        if key.endswith("/" + requested_key)
        or PurePosixPath(key).name == PurePosixPath(requested_key).name
    ]
    if len(suffix_matches) == 1:
        return suffix_matches[0]
    raise KeyError(
        f"ZIP member not found or ambiguous: {requested}; "
        f"matches={len(suffix_matches)}"
    )


def find_column(columns, aliases):
    # Locate common inventory fields without relying on fixed Excel columns.
    alias_keys = {normalise_name(alias) for alias in aliases}
    matches = [
        column
        for column in columns
        if normalise_name(column) in alias_keys
    ]
    return matches[0] if len(matches) == 1 else None


def classify_filename(file_name):
    # Return the dataset family and year when the name matches exactly.
    base_name = PurePosixPath(str(file_name).replace("\\", "/")).name
    for family, pattern in FILE_FAMILIES.items():
        match = pattern.fullmatch(base_name)
        if match:
            year = int(match.group(1))
            if START_YEAR <= year <= END_YEAR:
                return family, year
    return None


def parse_inventory_location(location):
    # Inventory locations use: nested archive :: CSV member.
    parts = [
        part.strip()
        for part in re.split(r"\s*::\s*", str(location))
        if part.strip()
    ]
    if len(parts) < 2:
        raise ValueError(
            "Expected 'nested archive :: CSV member' in file_location; "
            f"received: {location}"
        )
    return parts[0], parts[-1]


def choose_run_sheet(excel_file, run_code):
    # Match sheet names such as Run_G0001 without relying on sheet order.
    target = normalise_name(run_code)
    matches = [
        sheet
        for sheet in excel_file.sheet_names
        if normalise_name(sheet) == target
    ]
    if len(matches) != 1:
        raise KeyError(
            f"Expected one inventory sheet for {run_code}; found {matches}"
        )
    return matches[0]


def scan_csv_member(zip_file, member, chunk_rows):
    # Read each CSV once in chunks to determine exact rows, columns and encoding.
    errors = []
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            resolved = resolve_zip_member(zip_file, member)
            row_count = 0
            source_columns = None
            with zip_file.open(resolved) as stream:
                reader = pd.read_csv(
                    stream,
                    encoding=encoding,
                    chunksize=chunk_rows,
                    low_memory=False,
                )
                for chunk in reader:
                    if source_columns is None:
                        source_columns = list(chunk.columns)
                    row_count += len(chunk)
            if source_columns is None:
                # Header-only CSV: read the header separately.
                with zip_file.open(resolved) as stream:
                    empty = pd.read_csv(
                        stream,
                        encoding=encoding,
                        nrows=0,
                    )
                source_columns = list(empty.columns)
            return {
                "resolved_member": resolved,
                "encoding": encoding,
                "row_count": int(row_count),
                "source_columns": source_columns,
            }
        except UnicodeDecodeError as error:
            errors.append(f"{encoding}: {error}")
    raise UnicodeError(
        f"Could not decode {member}. Attempts: {' | '.join(errors)}"
    )


def iter_csv_chunks(zip_file, member, encoding, chunk_rows):
    # Yield chunks using the successful encoding recorded during the scan pass.
    resolved = resolve_zip_member(zip_file, member)
    with zip_file.open(resolved) as stream:
        yield from pd.read_csv(
            stream,
            encoding=encoding,
            chunksize=chunk_rows,
            low_memory=False,
        )


def safe_excel_value(value):
    # Convert pandas/numpy missing values and scalars into XlsxWriter-safe values.
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, str) and len(value) > 32_767:
        return value[:32_767]
    return value


def record_validation(records, check, passed, actual, expected, hard=True):
    # Save every test to a CSV audit and stop later when a hard check fails.
    records.append(
        {
            "check": check,
            "status": "PASS" if bool(passed) else "FAIL",
            "hard_check": bool(hard),
            "actual": str(actual),
            "expected": str(expected),
        }
    )


In [ ]:
# STEP 5 (Select and validate all inventory records)

validation_records = []

record_validation(
    validation_records,
    "Transfer ZIP exists",
    RAW_TRANSFER_ZIP.is_file(),
    RAW_TRANSFER_ZIP,
    "Existing file",
)
record_validation(
    validation_records,
    "Combined inventory exists",
    FILE_INVENTORY_PATH.is_file(),
    FILE_INVENTORY_PATH,
    "Existing file",
)
if not RAW_TRANSFER_ZIP.is_file():
    raise FileNotFoundError(RAW_TRANSFER_ZIP)
if not FILE_INVENTORY_PATH.is_file():
    raise FileNotFoundError(FILE_INVENTORY_PATH)

inventory_book = pd.ExcelFile(FILE_INVENTORY_PATH)
selected_frames = []

for run_id, run_code in zip(RUN_IDS, RUN_CODES):
    sheet_name = choose_run_sheet(inventory_book, run_code)
    run_inventory = pd.read_excel(
        FILE_INVENTORY_PATH,
        sheet_name=sheet_name,
    )
    file_name_column = find_column(
        run_inventory.columns,
        ("file_name", "filename", "file name"),
    )
    location_column = find_column(
        run_inventory.columns,
        ("file_location", "location", "file path", "path"),
    )
    if file_name_column is None or location_column is None:
        raise KeyError(
            f"{run_code}: inventory requires file_name and file_location. "
            f"Columns={list(run_inventory.columns)}"
        )

    selected_rows = []
    for source_index, row in run_inventory.iterrows():
        classification = classify_filename(row[file_name_column])
        if classification is None:
            continue
        family, year = classification
        inner_archive, csv_member = parse_inventory_location(
            row[location_column]
        )
        selected_rows.append(
            {
                "run_id": run_id,
                "run_code": run_code,
                "inventory_sheet": sheet_name,
                "inventory_excel_row": int(source_index) + 2,
                "dataset_family": family,
                "year": year,
                "file_name": str(row[file_name_column]).strip(),
                "file_location": str(row[location_column]).strip(),
                "inner_archive_member": inner_archive,
                "csv_member": csv_member,
            }
        )

    selected_run = pd.DataFrame(selected_rows)
    if selected_run.empty:
        raise ValueError(f"{run_code}: no target biodiversity files found.")

    duplicate_keys = selected_run.duplicated(
        ["dataset_family", "year"],
        keep=False,
    )
    actual_keys = set(
        zip(selected_run["dataset_family"], selected_run["year"])
    )
    record_validation(
        validation_records,
        f"{run_code} has no duplicate family/year files",
        not duplicate_keys.any(),
        selected_run.loc[
            duplicate_keys,
            ["dataset_family", "year", "file_name"],
        ].to_dict("records"),
        "No duplicates",
    )
    record_validation(
        validation_records,
        f"{run_code} has all seven families for 2025-2050",
        actual_keys == EXPECTED_FILE_KEYS,
        {
            "found": len(actual_keys),
            "missing": sorted(EXPECTED_FILE_KEYS - actual_keys),
            "extra": sorted(actual_keys - EXPECTED_FILE_KEYS),
        },
        f"{EXPECTED_FILES_PER_RUN} exact family/year combinations",
    )
    record_validation(
        validation_records,
        f"{run_code} uses one nested Run archive",
        selected_run["inner_archive_member"].nunique() == 1,
        selected_run["inner_archive_member"].unique().tolist(),
        "One archive member",
    )
    selected_frames.append(selected_run)

inventory_selection_df = pd.concat(
    selected_frames,
    ignore_index=True,
    sort=False,
).sort_values(
    ["run_id", "year", "dataset_family"],
    kind="mergesort",
).reset_index(drop=True)

hard_failures = [
    item for item in validation_records
    if item["hard_check"] and item["status"] != "PASS"
]
if hard_failures:
    raise AssertionError(
        "Inventory coverage validation failed:\n"
        + pd.DataFrame(hard_failures).to_string(index=False)
    )

inventory_selection_df.to_csv(
    INVENTORY_SELECTION_CSV,
    index=False,
)

print(
    f"Selected {len(inventory_selection_df):,} files "
    f"({EXPECTED_FILES_PER_RUN} per run)."
)
display(
    inventory_selection_df.groupby(
        ["run_code", "dataset_family"]
    ).size().unstack(fill_value=0)
)


In [ ]:
# STEP 6 - Build one workbook per scenario run and one sheet per year

source_audit_rows = []
sheet_audit_rows = []
workbook_audit_rows = []

with ZipFile(RAW_TRANSFER_ZIP) as outer_zip:
    for run_id, run_code in zip(RUN_IDS, RUN_CODES):
        run_selection = inventory_selection_df.loc[
            inventory_selection_df["run_code"].eq(run_code)
        ].copy()
        inner_request = run_selection[
            "inner_archive_member"
        ].iloc[0]
        temporary_archive = TEMP_DIR / f"{run_code}_Run_Archive.zip"
        final_workbook_path = RUN_WORKBOOK_PATHS[run_code]
        partial_workbook_path = (
            TEMP_DIR / f"SNES-ECNES-{run_code}.partial.xlsx"
        )

        # Remove only a previous incomplete file for this run.
        if partial_workbook_path.exists():
            partial_workbook_path.unlink()

        workbook = None
        completed_successfully = False
        try:
            resolved_inner = resolve_zip_member(
                outer_zip,
                inner_request,
            )
            archive_info = outer_zip.getinfo(resolved_inner)
            free_bytes = shutil.disk_usage(TEMP_DIR).free
            required_bytes = int(archive_info.file_size * 1.08) + 50_000_000
            if free_bytes < required_bytes:
                raise OSError(
                    f"{run_code}: insufficient temporary capacity. "
                    f"Need about {required_bytes / 1024**3:.2f} GB; "
                    f"available {free_bytes / 1024**3:.2f} GB."
                )

            with outer_zip.open(resolved_inner) as source:
                with temporary_archive.open("wb") as destination:
                    shutil.copyfileobj(
                        source,
                        destination,
                        length=32 * 1024 * 1024,
                    )
            if temporary_archive.stat().st_size != archive_info.file_size:
                raise IOError(
                    f"{run_code}: temporary archive copy is incomplete."
                )

            with ZipFile(temporary_archive) as run_zip:
                scan_records = []

                # First pass: determine exact rows, columns and encoding.
                for record in run_selection.itertuples(index=False):
                    scan = scan_csv_member(
                        run_zip,
                        record.csv_member,
                        CSV_CHUNK_ROWS,
                    )
                    scan_records.append(
                        {
                            **record._asdict(),
                            **scan,
                        }
                    )

                # Validate every year before creating the workbook.
                year_plans = {}
                for year in EXPECTED_YEARS:
                    year_scans = [
                        item for item in scan_records
                        if item["year"] == year
                    ]
                    union_source_columns = []
                    seen_columns = set()
                    for scan in year_scans:
                        for column in scan["source_columns"]:
                            if column not in seen_columns:
                                union_source_columns.append(column)
                                seen_columns.add(column)

                    total_year_rows = sum(
                        item["row_count"] for item in year_scans
                    )
                    all_columns = (
                        PROVENANCE_COLUMNS + union_source_columns
                    )
                    record_validation(
                        validation_records,
                        f"{run_code} {year} fits one Excel worksheet",
                        total_year_rows <= MAX_DATA_ROWS_PER_SHEET,
                        total_year_rows,
                        f"<= {MAX_DATA_ROWS_PER_SHEET:,}",
                    )
                    record_validation(
                        validation_records,
                        f"{run_code} {year} fits Excel column limit",
                        len(all_columns) <= MAX_EXCEL_COLUMNS,
                        len(all_columns),
                        f"<= {MAX_EXCEL_COLUMNS:,}",
                    )
                    record_validation(
                        validation_records,
                        f"{run_code} {year} contains seven source files",
                        len(year_scans) == len(FILE_FAMILIES),
                        len(year_scans),
                        len(FILE_FAMILIES),
                    )
                    if total_year_rows > MAX_DATA_ROWS_PER_SHEET:
                        raise OverflowError(
                            f"{run_code} year {year} has "
                            f"{total_year_rows:,} data rows, which cannot "
                            "fit in one Excel worksheet."
                        )
                    if len(all_columns) > MAX_EXCEL_COLUMNS:
                        raise OverflowError(
                            f"{run_code} year {year} has "
                            f"{len(all_columns):,} union columns, which "
                            "exceeds Excel's column limit."
                        )
                    if len(year_scans) != len(FILE_FAMILIES):
                        raise AssertionError(
                            f"{run_code} year {year}: expected "
                            f"{len(FILE_FAMILIES)} source files; "
                            f"found {len(year_scans)}."
                        )
                    year_plans[year] = {
                        "scan_records": year_scans,
                        "all_columns": all_columns,
                        "total_rows": total_year_rows,
                    }

                # Write-only mode streams rows to disk.
                workbook = Workbook(write_only=True)

                for year in EXPECTED_YEARS:
                    plan = year_plans[year]
                    year_scans = plan["scan_records"]
                    all_columns = plan["all_columns"]
                    total_year_rows = plan["total_rows"]

                    worksheet = workbook.create_sheet(title=str(year))
                    worksheet.sheet_view.showGridLines = True
                    worksheet.freeze_panes = "A2"
                    worksheet.auto_filter.ref = (
                        f"A1:{get_column_letter(len(all_columns))}"
                        f"{total_year_rows + 1}"
                    )
                    worksheet.row_dimensions[1].height = 32

                    # Set widths before streaming rows to a write-only sheet.
                    provenance_widths = {
                        "Run_ID": 10,
                        "Run_Name": 16,
                        "Dataset_Family": 32,
                        "File_Year": 12,
                        "Source_File": 48,
                        "Source_Member": 60,
                        "Source_Row": 12,
                    }
                    for column_index, column in enumerate(
                        all_columns,
                        start=1,
                    ):
                        width = provenance_widths.get(
                            column,
                            min(max(len(str(column)) + 2, 14), 32),
                        )
                        worksheet.column_dimensions[
                            get_column_letter(column_index)
                        ].width = width

                    # Create styled write-only header cells.
                    header_cells = []
                    for column in all_columns:
                        cell = WriteOnlyCell(worksheet, value=column)
                        cell.font = Font(
                            bold=True,
                            color="FFFFFFFF",
                        )
                        cell.fill = PatternFill(
                            fill_type="solid",
                            fgColor="FF1F4E78",
                        )
                        cell.alignment = Alignment(
                            horizontal="center",
                            vertical="center",
                            wrap_text=True,
                        )
                        header_cells.append(cell)
                    worksheet.append(header_cells)

                    row_cursor = 1

                    # Second pass: stream the seven files for this year.
                    for scan in year_scans:
                        source_row_cursor = 2
                        sheet_start_row = row_cursor + 1
                        rows_written_for_file = 0

                        for chunk in iter_csv_chunks(
                            run_zip,
                            scan["csv_member"],
                            scan["encoding"],
                            CSV_CHUNK_ROWS,
                        ):
                            chunk = chunk.copy()
                            chunk.insert(0, "Source_Row", np.arange(
                                source_row_cursor,
                                source_row_cursor + len(chunk),
                            ))
                            chunk.insert(
                                0,
                                "Source_Member",
                                scan["resolved_member"],
                            )
                            chunk.insert(
                                0,
                                "Source_File",
                                scan["file_name"],
                            )
                            chunk.insert(0, "File_Year", scan["year"])
                            chunk.insert(
                                0,
                                "Dataset_Family",
                                scan["dataset_family"],
                            )
                            chunk.insert(0, "Run_Name", run_code)
                            chunk.insert(0, "Run_ID", run_id)
                            chunk = chunk.reindex(columns=all_columns)

                            for values in chunk.itertuples(
                                index=False,
                                name=None,
                            ):
                                worksheet.append(
                                    [
                                        safe_excel_value(value)
                                        for value in values
                                    ]
                                )
                                row_cursor += 1
                                rows_written_for_file += 1
                            source_row_cursor += len(chunk)

                        if rows_written_for_file != scan["row_count"]:
                            raise AssertionError(
                                f"{run_code} {year} {scan['file_name']}: "
                                f"scanned {scan['row_count']:,} rows but "
                                f"wrote {rows_written_for_file:,}."
                            )

                        source_audit_rows.append(
                            {
                                "run_id": run_id,
                                "run_code": run_code,
                                "workbook_name": final_workbook_path.name,
                                "sheet_name": str(year),
                                "dataset_family": scan["dataset_family"],
                                "year": scan["year"],
                                "file_name": scan["file_name"],
                                "inner_archive_member": resolved_inner,
                                "csv_member": scan["resolved_member"],
                                "encoding": scan["encoding"],
                                "source_columns": len(
                                    scan["source_columns"]
                                ),
                                "source_rows": scan["row_count"],
                                "sheet_start_row": sheet_start_row,
                                "sheet_end_row": row_cursor,
                                "rows_written": rows_written_for_file,
                                "status": "PASS",
                            }
                        )

                    if row_cursor - 1 != total_year_rows:
                        raise AssertionError(
                            f"{run_code} {year}: expected "
                            f"{total_year_rows:,} worksheet rows; "
                            f"wrote {row_cursor - 1:,}."
                        )

                    sheet_audit_rows.append(
                        {
                            "run_id": run_id,
                            "run_code": run_code,
                            "workbook_name": final_workbook_path.name,
                            "sheet_name": str(year),
                            "year": year,
                            "source_files": len(year_scans),
                            "data_rows": total_year_rows,
                            "worksheet_rows_including_header": row_cursor,
                            "union_columns": len(all_columns),
                            "status": "PASS",
                        }
                    )

                workbook.save(partial_workbook_path)
                workbook.close()
                completed_successfully = True

            if (
                not completed_successfully
                or not partial_workbook_path.exists()
            ):
                raise RuntimeError(
                    f"{run_code}: temporary workbook did not finish."
                )

            # Replace this run's prior completed workbook only after all
            # six year sheets have been written successfully.
            partial_workbook_path.replace(final_workbook_path)
            workbook_audit_rows.append(
                {
                    "run_id": run_id,
                    "run_code": run_code,
                    "workbook_name": final_workbook_path.name,
                    "workbook_path": str(final_workbook_path),
                    "year_sheets": len(EXPECTED_YEARS),
                    "total_data_rows": sum(
                        plan["total_rows"]
                        for plan in year_plans.values()
                    ),
                    "size_bytes": final_workbook_path.stat().st_size,
                    "status": "PASS",
                }
            )
        except Exception:
            if workbook is not None:
                try:
                    workbook.close()
                except Exception:
                    pass
            if partial_workbook_path.exists():
                partial_workbook_path.unlink()
            raise
        finally:
            if temporary_archive.exists():
                temporary_archive.unlink()

        record_validation(
            validation_records,
            f"{run_code} temporary archive removed",
            not temporary_archive.exists(),
            temporary_archive,
            "No residual archive",
        )
        record_validation(
            validation_records,
            f"{run_code} workbook created",
            final_workbook_path.is_file()
            and final_workbook_path.stat().st_size > 10_000,
            final_workbook_path,
            "Existing workbook larger than 10 KB",
        )
        print(
            f"Completed {run_code}: "
            f"{workbook_audit_rows[-1]['total_data_rows']:,} rows "
            f"across {len(EXPECTED_YEARS)} year sheets"
        )

source_audit_df = pd.DataFrame(source_audit_rows).sort_values(
    ["run_id", "year", "dataset_family"],
    kind="mergesort",
).reset_index(drop=True)
sheet_audit_df = pd.DataFrame(sheet_audit_rows).sort_values(
    ["run_id", "year"],
    kind="mergesort",
).reset_index(drop=True)
workbook_audit_df = pd.DataFrame(workbook_audit_rows).sort_values(
    "run_id"
).reset_index(drop=True)

source_audit_df.to_csv(SOURCE_AUDIT_CSV, index=False)
sheet_audit_df.to_csv(SHEET_AUDIT_CSV, index=False)

display(workbook_audit_df)
display(sheet_audit_df)
print("Created", len(workbook_audit_df), "run workbooks in:", OUTPUT_DIR)

In [ ]:
# Step 7 - Validate the 16 completed workbooks

record_validation(
    validation_records,
    "Exactly 16 expected run workbooks were created",
    all(
        path.is_file() and path.stat().st_size > 10_000
        for path in RUN_WORKBOOK_PATHS.values()
    ),
    {
        run_code: {
            "exists": path.is_file(),
            "size_bytes": path.stat().st_size if path.exists() else 0,
        }
        for run_code, path in RUN_WORKBOOK_PATHS.items()
    },
    "16 expected workbooks, each larger than 10 KB",
)

# Write-only sheets do not store cached max_row/max_column dimensions.
# Inspect the saved headers and use the streaming audits to reconcile rows.
workbook_header_rows = []
for run_code in RUN_CODES:
    workbook_path = RUN_WORKBOOK_PATHS[run_code]
    workbook_check = load_workbook(
        workbook_path,
        read_only=True,
        data_only=True,
    )
    actual_sheet_names = workbook_check.sheetnames
    record_validation(
        validation_records,
        f"{run_code} has exactly the six year sheets",
        actual_sheet_names == YEAR_SHEET_NAMES,
        actual_sheet_names,
        YEAR_SHEET_NAMES,
    )

    for year in EXPECTED_YEARS:
        sheet_name = str(year)
        worksheet = workbook_check[sheet_name]
        header_values = next(
            worksheet.iter_rows(
                min_row=1,
                max_row=1,
                values_only=True,
            ),
            tuple(),
        )
        expected = sheet_audit_df.loc[
            sheet_audit_df["run_code"].eq(run_code)
            & sheet_audit_df["year"].eq(year)
        ].iloc[0]
        workbook_header_rows.append(
            {
                "run_code": run_code,
                "year": year,
                "sheet_name": sheet_name,
                "saved_header_columns": len(header_values),
                "expected_columns": int(expected["union_columns"]),
                "provenance_prefix": list(
                    header_values[:len(PROVENANCE_COLUMNS)]
                ),
            }
        )
    workbook_check.close()

workbook_headers_df = pd.DataFrame(workbook_header_rows)
record_validation(
    validation_records,
    "Every saved year sheet has its complete union schema",
    (
        workbook_headers_df["saved_header_columns"]
        == workbook_headers_df["expected_columns"]
    ).all(),
    int(
        (
            workbook_headers_df["saved_header_columns"]
            != workbook_headers_df["expected_columns"]
        ).sum()
    ),
    "0 header-width mismatches",
)
record_validation(
    validation_records,
    "Every saved year sheet starts with provenance columns",
    workbook_headers_df["provenance_prefix"].apply(
        lambda values: values == PROVENANCE_COLUMNS
    ).all(),
    int(
        (~workbook_headers_df["provenance_prefix"].apply(
            lambda values: values == PROVENANCE_COLUMNS
        )).sum()
    ),
    "0 provenance-header mismatches",
)
record_validation(
    validation_records,
    "Every year sheet audit has one header plus all streamed rows",
    (
        sheet_audit_df["worksheet_rows_including_header"]
        == sheet_audit_df["data_rows"] + 1
    ).all(),
    int(
        (
            sheet_audit_df["worksheet_rows_including_header"]
            != sheet_audit_df["data_rows"] + 1
        ).sum()
    ),
    "0 row-count mismatches",
)
record_validation(
    validation_records,
    "Every source file reconciles scanned and written rows",
    (
        source_audit_df["source_rows"]
        == source_audit_df["rows_written"]
    ).all(),
    int(
        (
            source_audit_df["source_rows"]
            != source_audit_df["rows_written"]
        ).sum()
    ),
    "0 source-row mismatches",
)
record_validation(
    validation_records,
    "Every run-year contains seven source files",
    source_audit_df.groupby(
        ["run_code", "year"]
    ).size().eq(len(FILE_FAMILIES)).all(),
    source_audit_df.groupby(
        ["run_code", "year"]
    ).size().to_dict(),
    len(FILE_FAMILIES),
)
record_validation(
    validation_records,
    "Every run contains seven file families",
    source_audit_df.groupby("run_code")[
        "dataset_family"
    ].nunique().eq(len(FILE_FAMILIES)).all(),
    source_audit_df.groupby("run_code")[
        "dataset_family"
    ].nunique().to_dict(),
    len(FILE_FAMILIES),
)
record_validation(
    validation_records,
    "Every run contains all expected years",
    source_audit_df.groupby("run_code")["year"].apply(
        lambda values: sorted(values.unique().tolist()) == EXPECTED_YEARS
    ).all(),
    source_audit_df.groupby("run_code")["year"].apply(
        lambda values: sorted(values.unique().tolist())
    ).to_dict(),
    EXPECTED_YEARS,
)

residual_archives = list(TEMP_DIR.glob("*Run_Archive.zip"))
residual_partials = list(TEMP_DIR.glob("*.partial.xlsx"))
record_validation(
    validation_records,
    "No temporary Run archives remain",
    not residual_archives,
    [path.name for path in residual_archives],
    "None",
)
record_validation(
    validation_records,
    "No partial Excel workbooks remain",
    not residual_partials,
    [path.name for path in residual_partials],
    "None",
)

validation_df = pd.DataFrame(validation_records)
validation_df.to_csv(VALIDATION_CSV, index=False)
display(validation_df)

failed_hard_checks = validation_df.loc[
    validation_df["hard_check"]
    & validation_df["status"].ne("PASS")
]
if not failed_hard_checks.empty:
    raise AssertionError(
        "One or more hard validations failed:\n"
        + failed_hard_checks.to_string(index=False)
    )

print("All hard validations passed.")
print("Run workbooks:", OUTPUT_DIR)
print("Inventory selection:", INVENTORY_SELECTION_CSV)
print("Source audit:", SOURCE_AUDIT_CSV)
print("Sheet audit:", SHEET_AUDIT_CSV)
print("Validation audit:", VALIDATION_CSV)